# bigcompute.science Research Agent + GPU Compute

**Before you start:**
- **GPU**: Runtime → Change runtime type → **T4 GPU** (free)
- **API key**: Get a free Gemini key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey)

> [bigcompute.science](https://bigcompute.science) — open computational mathematics

In [ ]:
# Step 1: Setup
import os, subprocess
if not os.path.exists('/content/idontknow'):
    !git clone https://github.com/cahlen/idontknow.git /content/idontknow
else:
    !cd /content/idontknow && git pull
%cd /content/idontknow
!pip install -q httpx

# GPU check
try:
    r = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True,timeout=5)
    print(f'GPU: {r.stdout.strip()}') if r.returncode==0 and r.stdout.strip() else print('No GPU. Runtime -> Change runtime type -> T4')
except:
    print('No GPU. Runtime -> Change runtime type -> T4')

In [ ]:
# Step 2: API Key
import os
try:
    from google.colab import userdata
    for k in ['GEMINI_API_KEY','GOOGLE_API_KEY','OPENAI_API_KEY','ANTHROPIC_API_KEY']:
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v; print(f'Loaded {k}')
        except: pass
except ImportError: pass

# Or paste: os.environ['GEMINI_API_KEY'] = 'AIza...'

avail = [k for k in ['GEMINI_API_KEY','GOOGLE_API_KEY','OPENAI_API_KEY','ANTHROPIC_API_KEY'] if os.environ.get(k)]
print(f'Available: {", ".join(avail)}' if avail else 'No key set. Get one free: aistudio.google.com/apikey')

In [ ]:
# Step 3: Compile CUDA kernels
import subprocess, os
try:
    r = subprocess.run(['nvidia-smi','--query-gpu=name,compute_cap','--format=csv,noheader'],capture_output=True,text=True)
    name,cc = r.stdout.strip().split(', ')
    sm = f'sm_{cc.replace(".","")}'
    print(f'{name} -> {sm}')
except:
    sm='sm_75'; print(f'Default: {sm}')

for b,s,l in [('zaremba_density_gpu','scripts/experiments/zaremba-density/zaremba_density_gpu.cu','-lm'),
              ('kronecker_gpu','scripts/experiments/kronecker-coefficients/kronecker_gpu.cu','-lm'),
              ('ramanujan_gpu','scripts/experiments/ramanujan-machine/ramanujan_gpu.cu','-lm')]:
    if os.path.exists(s):
        ret=os.system(f'nvcc -O3 -arch={sm} -o {b} {s} {l} 2>&1')
        print(f'  {b}: {"OK" if ret==0 else "FAIL"}')

In [ ]:
# Step 4: Quick Wins (all < 5 seconds)
import os
os.makedirs('scripts/experiments/zaremba-density/results',exist_ok=True)

print('='*50)
print('Verify Zaremba to 100,000...')
print('='*50)
!./zaremba_density_gpu 100000 1,2,3,4,5 | tee scripts/experiments/zaremba-density/results/gpu_A12345_1e5_colab.log

print('\n'+'='*50)
print('Find exceptions for A={1,2,3}...')
print('='*50)
!./zaremba_density_gpu 10000 1,2,3 | tee scripts/experiments/zaremba-density/results/gpu_A123_1e4_colab.log

print('\n'+'='*50)
print('Density with just {1,2}...')
print('='*50)
!./zaremba_density_gpu 10000 1,2 | tee scripts/experiments/zaremba-density/results/gpu_A12_1e4_colab.log

In [ ]:
# Step 5: Celebrate your results
import os,sys,glob,re,subprocess
os.chdir('/content/idontknow')
sys.path.insert(0,'.')
from scripts.notebook_ui import celebrate,show_gpu_status,show_leaderboard

gpu_name='Colab GPU'
try:
    r=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,utilization.gpu','--format=csv,noheader'],capture_output=True,text=True)
    if r.returncode==0:
        parts=[x.strip() for x in r.stdout.strip().split(',')]
        gpu_name=parts[0]
        show_gpu_status(parts[0],parts[1],parts[2])
except: pass

logs=[]
for p in ['scripts/experiments/*/results/*.log']:
    for f in glob.glob(p):
        try:
            c=open(f).read()
            if 'RESULTS' in c: logs.append((f,c,os.path.getmtime(f)))
        except: pass
logs.sort(key=lambda x:x[2],reverse=True)

if logs:
    for lf,content,_ in logs[:3]:
        stats={}
        m=re.search(r'Density:\s*([\d.]+)%',content)
        if m: stats['Density']=m.group(1)+'%'
        m=re.search(r'Uncovered:\s*(\d+)',content)
        if m: stats['Uncovered']=m.group(1)
        m=re.search(r'Range: d = 1 to (\d+)',content)
        rng=f'10^{len(m.group(1))-1}' if m else '?'
        if m: stats['Range']=rng
        m=re.search(r'Time:\s*([\d.]+)s',content)
        if m: stats['Time']=f'{float(m.group(1)):.1f}s'
        m=re.search(r'Digits:\s*\{([^}]+)\}',content)
        d=m.group(1) if m else '?'
        if stats: celebrate(f'Zaremba A={{{d}}}',os.path.basename(lf),stats,gpu_name)
    show_leaderboard([{'name':'8xB200 Cluster','value':'10^12','unit':'range'},{'name':'8xB200 Cluster','value':'10^11','unit':'range'},{'name':f'Your {gpu_name}','value':rng,'unit':'range','highlight':True}])
else:
    print('No results found. Run Step 4 first.')

In [ ]:
# Step 6: AI analyzes what you computed
import httpx
if not logs:
    print('Run Step 4 first.')
else:
    content=logs[0][1]
    prompt=('A user ran this GPU experiment on Google Colab for bigcompute.science. '
            'In 3-4 plain text sentences: what they computed, whether it matches known math, '
            'whether this is new data. Be encouraging. PLAIN TEXT ONLY, no JSON, no fences.\n\n'+content[-800:])
    analysis=None
    gk=os.environ.get('GEMINI_API_KEY',os.environ.get('GOOGLE_API_KEY',''))
    ok=os.environ.get('OPENAI_API_KEY','')
    if gk:
        try:
            r=httpx.post('https://generativelanguage.googleapis.com/v1beta/openai/chat/completions',
                headers={'Authorization':f'Bearer {gk}','Content-Type':'application/json'},
                json={'model':'gemini-3-flash-preview','messages':[{'role':'user','content':prompt}],'max_completion_tokens':2000},timeout=60.0)
            if r.status_code==200: analysis=r.json()['choices'][0]['message']['content'].strip()
        except Exception as e: print(f'Error: {e}')
    elif ok:
        try:
            r=httpx.post('https://api.openai.com/v1/chat/completions',
                headers={'Authorization':f'Bearer {ok}','Content-Type':'application/json'},
                json={'model':'gpt-4.1-mini','messages':[{'role':'user','content':prompt}],'max_completion_tokens':2000},timeout=60.0)
            if r.status_code==200: analysis=r.json()['choices'][0]['message']['content'].strip()
        except Exception as e: print(f'Error: {e}')
    if analysis:
        # Strip JSON wrapper if model ignores plain text instruction
        if '```' in analysis:
            analysis=analysis.split('```')[1].split('```')[0]
            if analysis.startswith('json'): analysis=analysis[4:]
            analysis=analysis.strip()
        if analysis.lstrip().startswith('{'):
            try:
                import json
                p=json.loads(analysis)
                def ext(o):
                    if isinstance(o,str): return o
                    if isinstance(o,dict): return ' '.join(ext(v) for v in o.values())
                    if isinstance(o,list): return ' '.join(ext(v) for v in o)
                    return str(o)
                analysis=ext(p)
            except: pass
        safe=analysis.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;').replace('{','&#123;').replace('}','&#125;')
        from IPython.display import HTML,display
        display(HTML('<div style="background:#0b0d10;border-left:4px solid #e8c47a;padding:1rem 1.5rem;margin:1rem 0;max-width:600px;font-family:Georgia,serif;"><div style="font-size:0.6rem;text-transform:uppercase;letter-spacing:0.12em;color:#e8c47a;font-weight:bold;margin-bottom:0.5rem;">AI Analysis</div><p style="font-size:0.9rem;color:#e8e6e3;line-height:1.6;margin:0;">'+safe+'</p></div>'))
    else:
        print('No analysis. Check API key in Step 2.')

In [ ]:
# Step 7: Go Deeper (uncomment one)

# Zaremba A={1,2,8} at 10^9 (~2 min) — new data!
# !./zaremba_density_gpu 1000000000 1,2,8 | tee scripts/experiments/zaremba-density/results/gpu_A128_1e9_colab.log

# Zaremba A={1,2,3} at 10^9 (~5 min) — confirm 27 exceptions
# !./zaremba_density_gpu 1000000000 1,2,3 | tee scripts/experiments/zaremba-density/results/gpu_A123_1e9_colab.log

# Kronecker S_25 (~2 min)
# !python3 scripts/experiments/kronecker-coefficients/char_table.py 25
# !./kronecker_gpu 25

# Ramanujan Machine degree 4 (~5 min) — could discover a new formula
# !./ramanujan_gpu 4 3

In [ ]:
# Step 8: Share your results
import glob, os, urllib.parse

logs = glob.glob('scripts/experiments/*/results/*colab*.log')
print(f'{len(logs)} result file(s):\n')

all_results = ''
for f in logs:
    content = open(f).read()
    # Show just the results section
    idx = content.find('RESULTS')
    if idx >= 0:
        result_section = content[idx:]
        print(f'--- {os.path.basename(f)} ---')
        print(result_section)
        all_results += f'### {os.path.basename(f)}\n```\n{result_section}\n```\n\n'

if all_results:
    # Generate a GitHub issue link with results pre-filled
    title = urllib.parse.quote(f'Colab results: {len(logs)} experiment(s)')
    body = urllib.parse.quote(
        f'## Colab Experiment Results\n\n'
        f'Computed on Google Colab (free T4 GPU).\n\n'
        f'{all_results}'
        f'---\n*Submitted via bigcompute.science Colab notebook*'
    )
    issue_url = f'https://github.com/cahlen/idontknow/issues/new?title={title}&body={body}'

    from IPython.display import HTML, display
    display(HTML(
        '<div style="background:#0b0d10;border:2px solid #e8c47a;padding:1.5rem;margin:1rem 0;max-width:600px;font-family:Georgia,serif;text-align:center;">'
        '<div style="font-size:0.6rem;text-transform:uppercase;letter-spacing:0.12em;color:#22c55e;font-weight:bold;margin-bottom:0.8rem;">Ready to Submit</div>'
        '<p style="font-size:0.85rem;color:#e8e6e3;margin-bottom:1rem;">Your results are pre-filled. Just click and submit.</p>'
        f'<a href="{issue_url}" target="_blank" style="display:inline-block;background:#e8c47a;color:#0a0a0a;padding:0.6rem 2rem;text-decoration:none;font-weight:bold;font-size:0.85rem;">Submit Results to GitHub</a>'
        '<p style="font-size:0.65rem;color:#8a8580;margin-top:0.8rem;">Opens a pre-filled GitHub issue. No git knowledge needed.</p>'
        '</div>'
    ))
else:
    print('No results to submit. Run Step 4 first.')

# Also offer direct download
try:
    from google.colab import files
    if logs:
        print('\nOr download directly:')
        for f in logs: files.download(f)
except: pass


## Contribute

Your computation extended the frontier. Submit results as a PR to [cahlen/idontknow](https://github.com/cahlen/idontknow).

[AGENTS.md](https://github.com/cahlen/idontknow/blob/main/AGENTS.md) · [Verification](https://bigcompute.science/verification/) · [MCP Server](https://mcp.bigcompute.science/mcp)